# Multi-agent systems

In [53]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", # "gemini-2.5-flash-lite" for cheaper use, "gemini-2.5-flash" for normal use
    vertexai=True
)

## Prepare SQLite DB for later usage
We use an pre-defined SQL to create tables and insert data to the tables.

## Prepare Agents
We will have two agents to form the multi-agents system
- Accommodation Booking Agent
- Travel Info Agent

#### Setup tools for Agents
Two tools for agents:
- Hotel booking tool
- B&B booking tool

#### Hotel Booking Tool
It is the tool from SQLite DB. 

In [54]:
### import packages
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [55]:
### instantiate the SQLite database
hotel_db = SQLDatabase.from_uri("sqlite:///hotel_db/cornwall_hotels.db")

### create an instance of SQLite database toolkit
hotel_db_toolkit = SQLDatabaseToolkit(db=hotel_db, llm=llm)

### get all the tools from hotel_db_toolkit
hotel_db_toolkit_tools = hotel_db_toolkit.get_tools()

#### B&B booking tool
A tool simulates calling a Restful booking API.

In [56]:
from typing import Annotated, Sequence, TypedDict, Literal, Optional, List, Dict

In [57]:
class BnBOffer(TypedDict): #A
    bnb_id: int
    bnb_name: str
    town: str
    available_rooms: int
    price_per_room: float

In [58]:
class BnBBookingService: #B
    @staticmethod
    def get_offers_near_town(town: str, num_rooms: int) \
        -> List[BnBOffer]: #C
        # Mocked REST API response: multiple BnBs per destination
        mock_bnb_offers = [ #D
            # Newquay
            {"bnb_id": 1, "bnb_name": "Seaside BnB", 
            "town": "Newquay", "available_rooms": 3, 
            "price_per_room": 80.0},
            {"bnb_id": 2, "bnb_name": "Surfside Guesthouse", 
            "town": "Newquay", "available_rooms": 2, 
            "price_per_room": 85.0},
            # Falmouth
            {"bnb_id": 3, "bnb_name": "Harbour View BnB", 
            "town": "Falmouth", "available_rooms": 4, 
            "price_per_room": 78.0},
            {"bnb_id": 4, "bnb_name": "Seafarer's Rest", 
            "town": "Falmouth", "available_rooms": 1, 
            "price_per_room": 90.0},
            # St Austell
            {"bnb_id": 5, "bnb_name": "Garden Gate BnB", 
            "town": "St Austell", "available_rooms": 2, "price_per_room": 82.0},
            {"bnb_id": 6, "bnb_name": "Coastal Cottage BnB", 
            "town": "St Austell", "available_rooms": 3, "price_per_room": 88.0},
            # Penzance
            {"bnb_id": 7, "bnb_name": "Penzance Pier BnB", 
            "town": "Penzance", "available_rooms": 2, "price_per_room": 95.0},
            {"bnb_id": 8, "bnb_name": "Cornish Charm BnB", 
            "town": "Penzance", "available_rooms": 3, "price_per_room": 87.0},
            # Camborne
            {"bnb_id": 9, "bnb_name": "Camborne Corner BnB", 
            "town": "Camborne", "available_rooms": 2, "price_per_room": 75.0},
            {"bnb_id": 10, "bnb_name": "Rose Cottage BnB", 
            "town": "Camborne", "available_rooms": 2, "price_per_room": 79.0},
            # Hayle
            {"bnb_id": 11, "bnb_name": "Hayle Haven BnB", 
            "town": "Hayle", "available_rooms": 3, "price_per_room": 83.0},
            {"bnb_id": 12, "bnb_name": "Dune View BnB", 
            "town": "Hayle", "available_rooms": 1, "price_per_room": 81.0},
            # Land's End
            {"bnb_id": 13, "bnb_name": "Land's End Lookout BnB", 
            "town": "Land's End", "available_rooms": 2, "price_per_room": 100.0},
            {"bnb_id": 14, "bnb_name": "Atlantic Edge BnB", 
            "town": "Land's End", "available_rooms": 2, "price_per_room": 105.0},
            # Bude
            {"bnb_id": 15, "bnb_name": "Bude Beach BnB", 
            "town": "Bude", "available_rooms": 2, "price_per_room": 77.0},
            {"bnb_id": 16, "bnb_name": "Cliffside BnB", 
            "town": "Bude", "available_rooms": 3, "price_per_room": 80.0},
            # Padstow
            {"bnb_id": 17, "bnb_name": "Padstow Harbour BnB", 
            "town": "Padstow", "available_rooms": 2, "price_per_room": 92.0},
            {"bnb_id": 18, "bnb_name": "Fisherman's Rest BnB", 
            "town": "Padstow", "available_rooms": 2, "price_per_room": 89.0},
            # St Ives
            {"bnb_id": 19, "bnb_name": "St Ives Bay BnB", "town": "St Ives", "available_rooms": 3, "price_per_room": 97.0},
            {"bnb_id": 20, "bnb_name": "Artists' Retreat BnB", "town": "St Ives", "available_rooms": 2, "price_per_room": 102.0},
            # Looe
            {"bnb_id": 21, "bnb_name": "Looe Riverside BnB", "town": "Looe", "available_rooms": 2, "price_per_room": 84.0},
            {"bnb_id": 22, "bnb_name": "Harbour Lights BnB", "town": "Looe", "available_rooms": 2, "price_per_room": 86.0},
            # Polperro
            {"bnb_id": 23, "bnb_name": "Polperro Cove BnB", "town": "Polperro", "available_rooms": 2, "price_per_room": 91.0},
            {"bnb_id": 24, "bnb_name": "Smuggler's Rest BnB", "town": "Polperro", "available_rooms": 2, "price_per_room": 93.0},
            # Mevagissey
            {"bnb_id": 25, "bnb_name": "Mevagissey Harbour BnB", "town": "Mevagissey", "available_rooms": 2, "price_per_room": 90.0},
            {"bnb_id": 26, "bnb_name": "Seafarer's BnB", "town": "Mevagissey", "available_rooms": 2, "price_per_room": 88.0},
            # Port Isaac
            {"bnb_id": 27, "bnb_name": "Port Isaac View BnB", 
            "town": "Port Isaac", "available_rooms": 2, 
            "price_per_room": 99.0},
            {"bnb_id": 28, "bnb_name": "Fisherman's Cottage BnB", 
            "town": "Port Isaac", "available_rooms": 2, 
            "price_per_room": 101.0},
            # Fowey
            {"bnb_id": 29, "bnb_name": "Fowey Quay BnB", 
            "town": "Fowey", "available_rooms": 2, 
            "price_per_room": 94.0},
            {"bnb_id": 30, "bnb_name": "Riverside Rest BnB", 
            "town": "Fowey", "available_rooms": 2, 
            "price_per_room": 96.0},
        ]
        offers = [offer for offer in 
            mock_bnb_offers 
            if offer["town"].lower() == town.lower() 
               and offer["available_rooms"] >= num_rooms]
        return offers
    
#A Define the return type of the BnB availability tool
#B Define the BnB availability tool
#C Call the BnB booking service to get the offers
#D Mocked BnB offers

Then define the tool for calling BnB service

In [59]:
from langchain_core.tools import tool

@tool(description="""Check BnB room availability and price for a destination Cornwall (UK).""")
def check_bnb_availability(destination:str, num_rooms:int) -> List[Dict]:
    offers = BnBBookingService.get_offers_near_town(destination, num_rooms)
    if not offers:
        return [{"error": f"No available BnBs found in {destination} for {num_rooms} rooms."}]
    return offers

### Create Accommodation Booking Agent

In [60]:
### Prepare the Agent State
import operator
from langgraph.managed.is_last_step import RemainingSteps
from langchain_core.messages import BaseMessage, HumanMessage

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps

In [61]:
from langgraph.prebuilt import create_react_agent

BOOKING_TOOLS = hotel_db_toolkit_tools + [check_bnb_availability]

accommodation_booking_agent = create_react_agent(
    model=llm,
    name="accommodation_booking_agent",
    tools=BOOKING_TOOLS,
    state_schema=AgentState,
    prompt="""
    You are a helpful assistant that can check hotel and BnB room availability and price for a destination in Cornwall(UK).
    You can use the tools to get the information you need. If the users does not specify the accommodation type, 
    you should check both hotels and BnBs.
    """
)

C:\Users\andrewchan\AppData\Local\Temp\ipykernel_25464\1339742426.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  accommodation_booking_agent = create_react_agent(


Let's try the newly build `accommodation_booking_agent`.

In [62]:
print("UK accommodation assistant (type 'exit' to quit)")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break
    state = {"messages": [HumanMessage(content=user_input)]}
    result = accommodation_booking_agent.invoke(state)

    response_msg = result["messages"][-1]
    print(f"Assistant: {response_msg.content}\n")

UK accommodation assistant (type 'exit' to quit)


You:  exit


### Travel Info Agent
We build a Travel Info Agent from `RAG-Agent` notebook. We could reuse tools by importing from prebuild package.

In [63]:
### restore the vectorstore
from langchain_community.vectorstores import Chroma
from langchain_google_vertexai import VertexAIEmbeddings

embedding = VertexAIEmbeddings(model="gemini-embedding-001")

persist_directory = "./chroma_langchain_db"
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

travelinfo_retriever = vectordb.as_retriever()

C:\Users\andrewchan\AppData\Local\Temp\ipykernel_25464\259449612.py:5: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding = VertexAIEmbeddings(model="gemini-embedding-001")


In [64]:
class WeatherForecast(TypedDict):
    town: str
    weather: Literal["sunny", "foggy", "rainy", "windy"]
    temperature: int

class WeatherForecastService:

    _weather_options = ["sunny", "foggy", "rainy", "windy"]
    _temp_min = 18
    _temp_max = 31

    @classmethod
    def get_forecast(cls, town: str) -> Optional[WeatherForecast]: #A
        weather = random.choice(cls._weather_options)
        temperature = random.randint(cls._temp_min, cls._temp_max)
        return WeatherForecast(town=town, weather=weather, temperature=temperature)

In [65]:
@tool(description="Search travel information about destinations in UK.")
def search_travel_info(query: str) -> str:
    """Search embedded WikiVoyage content for information about destinations in UK."""
    docs = ti_retriever.invoke(query)
    top = docs[:4] if isinstance(docs, list) else docs
    return "\n---\n".join(d.page_content for d in top)


@tool(description="Get the weather forecast, given a town name.")
def weather_forecast(town: str) -> dict:
    """Get a mock weather forecast for a given town. Returns a WeatherForecast object with weather and temperature."""
    forecast = WeatherForecastService.get_forecast(town)
    if forecast is None:
        return {"error": f"No weather data available for '{town}'."}
    return forecast

In [66]:
TRAVEL_INFO_TOOLS = [search_travel_info, weather_forecast]

travel_info_agent = create_react_agent(
    model=llm,
    name="travel_info_agent",
    tools=TRAVEL_INFO_TOOLS,
    state_schema=AgentState,
    prompt="""
    You are a helpful assistant that can search travel information and get the weather forecast.
    Only use the tools to find the information you need (including town names).
    """
)

C:\Users\andrewchan\AppData\Local\Temp\ipykernel_25464\2281479334.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  travel_info_agent = create_react_agent(


## Multi-Agent Architectures
I would demo two architectures which are:
- **router-based**
- **supervisor**

### Router-bases Multi-Agents
Here, we introduce a router agent which acts as an intelligent entry point. It receives the user's message, determines which specialized agents should handle the request, dispatch the task.
***
First step is to define the set of agents available for routing. We archieve this by declaring an enumertion for two agent types.

In [67]:
### AgentType Enum and Structured Output Model
from enum import Enum

class AgentType(str, Enum):
    travel_info_agent = "travel_info_agent"
    accommodation_booking_agent = "accommodation_booking_agent"

To ensure our router agent receives clear nd structured decisions from LLM, we define a `Pydantic` model that captures the LLM's output, i.e. <u>specifying which agent should handle each query</u>.

In [68]:
from pydantic import BaseModel, Field

class AgentTypeOutput(BaseModel): 
    agent: AgentType = Field(..., description="Which agent should handle the query?")

Create the router agent that with structured output.

In [69]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", # "gemini-2.5-flash-lite" for cheaper use, "gemini-2.5-flash" for normal use
    vertexai=True
)

llm_router = llm.with_structured_output(AgentTypeOutput)

#### Routing logic in the node

In [70]:
### Prepare the Agent State
import operator
from langgraph.managed.is_last_step import RemainingSteps
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps

In [37]:
ROUTER_SYSTEM_PROMPT = """
You are a router. Given the following user message, decide if it is a travel information question
(about destinations, attractions, or general travel info) or an accommodation booking question
(about hotels, BnBs, room availability, or prices).
If it is a travel information question, respond with 'travel_info_agent'. 
If it is an accommodation booking question, respond with 'accommodation_booking_agent'.
"""

In [40]:
from langgraph.types import Command

def router_agent_node(state: AgentState) -> Command[AgentType]:
    """Router node: decides which agent should handle the user query."""
    messages = state["messages"]
    last_msg = messages[-1] if messages else None
    if isinstance(last_msg, HumanMessage):
        user_input = last_msg.content
        router_messages = [
            SystemMessage(content=ROUTER_SYSTEM_PROMPT),
            HumanMessage(content=user_input)
        ]
        router_response = llm_router.invoke(router_messages)
        agent_name = router_response.agent.value
        return Command(update=state, goto=agent_name)
    # if not HumanMessage
    return Command(update=state, goto=AgentType.travel_info_agent)

#### Build the multi-agent graph

In [42]:
from langgraph.graph import StateGraph, END

In [44]:
router_graph = StateGraph(AgentState)

router_graph.add_node("router_agent", router_agent_node)
router_graph.add_node("travel_info_agent", travel_info_agent)
router_graph.add_node("accommodation_booking_agent", accommodation_booking_agent)

router_graph.add_edge("travel_info_agent", END)
router_graph.add_edge("accommodation_booking_agent", END)

router_graph.set_entry_point("router_agent")

travel_assistant_router = router_graph.compile()

In [46]:
print("UK accommodation assistant (type 'exit' to quit)")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break
    state = {"messages": [HumanMessage(content=user_input)]}
    result = travel_assistant_router.invoke(state)

    response_msg = result["messages"][-1]
    print(f"Assistant: {response_msg.content}\n")

UK accommodation assistant (type 'exit' to quit)


You:  Are there any hotel or BnB rooms available in Penzance?


Task router_agent with path ('__pregel_pull', 'router_agent') wrote to unknown channel remaining_steps, ignoring it.


Assistant: There are 2 rooms available at Penzance Pier BnB for £95 per room and 3 rooms available at Cornish Charm BnB for £87 per room.



You:  exit


### Multi-Agents with Supervisor 

In [71]:
from langgraph_supervisor.supervisor import create_supervisor

In [73]:
### the agents in the list under the field `agents` must have nam

travel_assistant_supervisor = create_supervisor(
    agents = [travel_info_agent, accommodation_booking_agent],
    model=llm,
    name="travel_assistant",
    prompt=(
        """
        You are a supervisor that manages two agents:  a travel information agent and an accommodation booking agent.
        You can answer user questions that might require calling both agents when needed.
        Decide which agent(s) to use for each user request and coordinate their responses.
        """
    )
).compile() #E


In [74]:
print("UK accommodation assistant (type 'exit' to quit)")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break
    state = {"messages": [HumanMessage(content=user_input)]}
    result = travel_assistant_supervisor.invoke(state)

    response_msg = result["messages"][-1]
    print(f"Assistant: {response_msg.content}\n")

UK accommodation assistant (type 'exit' to quit)


You:  Are there any hotel or BnB rooms available in Penzance?


Assistant: I can help you with that! Where in Cornwall are you looking to stay?



You:  anywhere is ok


Assistant: I can help you find accommodation in Cornwall. Since you mentioned "anywhere is ok", I'll look for both hotels and BnBs. Could you please tell me the destination you have in mind?



You:  exit
